In [ ]:
knitr::opts_chunk$set(echo = FALSE,message = FALSE,warning = FALSE)
options(scipen=999)


In [ ]:
if (!exists("book_load_packages", mode = "function")) source("scripts/book_runtime.R")
book_load_packages(fredr,
               dplyr,
               tidyr,
               lubridate,
               caret,
               partykit,
               pROC,
               devtools,
               httr,
               DT,
               httr,
               visNetwork,
               dkstat,
               data.table,
               pROC,
               rPref,
               progress,
               quantmod,
               rvest,
               ggrepel)
source(file = "money_theme.R")
source(file = "scripts/analysis/correlations.R")
book_correlation_namespace = "etf-correlations"


# ETF Tracking and Analysis

## Danish tax classification (2026) {#danish-fund-tax}

This catalogue covers ETFs and investment-fund certificates with a verified Yahoo Finance ticker and documented Danish tax treatment for 2026. The ordinary-depot columns describe a private investor with a regular taxable *aktiedepot*. **Realisationsbeskatning** taxes a gain or loss on sale; **lagerbeskatning** includes annual changes in value. Aktieindkomst and kapitalindkomst are separate income categories. Dividends and minimum distributions can be taxable before a sale. [Skattestyrelsen explains the fund categories and reporting](https://skat.dk/borger/aktier-og-andre-vaerdipapirer/skat-af-investeringsbeviser-udstedt-af-investeringsforeninger-og-investeringsselskaber).

| Documented fund status | Ordinary-depot gain/loss treatment |
|:--|:--|
| Aktiebaseret investeringsselskab on [Skattestyrelsen's 2026 list](https://skat.dk/erhverv/ekapital/vaerdipapirer/beviser-og-aktier-i-investeringsforeninger-og-selskaber-ifpa) | Lagerbeskatning · aktieindkomst |
| Other documented investeringsselskab | Lagerbeskatning · kapitalindkomst |
| Aktiebaseret investeringsinstitut med minimumsbeskatning | Realisationsbeskatning · aktieindkomst |
| Obligationsbaseret investeringsinstitut med minimumsbeskatning | Realisationsbeskatning · kapitalindkomst |

On an *aktiesparekonto*, eligible holdings instead follow the account's **17% annual lager taxation**, regardless of their ordinary-depot row. The `ask_eligible` field indicates whether the fund's documented category is eligible; your provider must also support holding that specific security in the account. An ETF is not eligible merely because it appears in a search result. See [Skattestyrelsen's aktiesparekonto rules](https://skat.dk/borger/aktier-og-andre-vaerdipapirer/aktiesparekonto).

The [2026 Positivliste CSV](https://github.com/ckrusemd/money-printer-go-brrr/blob/main/data/positivliste_2026.csv) is a dated copy of registered 2026 entries from Skattestyrelsen's workbook. The table below is the verified intersection with the two product lists, not every instrument on Positivlisten. The [SKAT securities-classification list](https://info.skat.dk/data.aspx?oid=2460620) provides the additional investment-fund status. A security is omitted when its tax status, account eligibility, or Yahoo symbol cannot be confirmed. These snapshots may become stale when SKAT or a fund changes its classification; confirm before trading or filing a return.
Absence from the table—especially for an ETF outside Positivlisten—is not itself a tax classification.


In [ ]:
fund_tax_catalogue <- read.csv(
  "data/investment_products_2026.csv",
  fileEncoding = "UTF-8",
  colClasses = "character",
  check.names = FALSE
)

display_tax <- fund_tax_catalogue %>%
  dplyr::transmute(
    Name = name,
    Type = type,
    ISIN = isin,
    `Yahoo Finance` = yahoo_ticker,
    Principle = ordinary_tax_principle,
    Income = ordinary_income_type,
    `Aktiesparekonto eligible` = ask_eligible,
    `Tax basis` = tax_basis
  )

DT::datatable(
  display_tax,
  rownames = FALSE,
  filter = "top",
  options = list(pageLength = 15, scrollX = TRUE, autoWidth = TRUE)
)


In [ ]:
df_etfs = openxlsx::read.xlsx(xlsxFile = "Chapters/05_ETFs/etf_list.xlsx")

get_yahoo_value = function(ticker, attempts = 3) {
  for (attempt in seq_len(attempts)) {
    prices = tryCatch(
      book_yahoo(ticker, attempts = 1L),
      error = function(error) NULL
    )

    if (!is.null(prices)) {
      return(
        prices %>%
      as.data.frame(.) %>%
      dplyr::mutate(Date=row.names(.)) %>%
      dplyr::mutate(Date=gsub("X","",Date)) %>%
      dplyr::mutate(Date=ymd(Date)) %>%
      dplyr::select(c(7,1)) %>%
      setNames(.,c("date","value")) %>%
      dplyr::mutate(series_id=ticker)# %>%
      # tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
      # tidyr::fill(value,.direction = "down")  %>%
      # tidyr::fill(series_id,.direction = "down")
      )
    }

    if (attempt < attempts) Sys.sleep(2 ^ (attempt - 1))
  }

  warning(sprintf("Skipping Yahoo Finance ticker %s after %d failed attempts", ticker, attempts))
  NULL
}

df_etf_yahoo = rbindlist(lapply(df_etfs$Ticker, get_yahoo_value), fill = TRUE)

if (!nrow(df_etf_yahoo) || all(is.na(df_etf_yahoo$value))) {
  stop("No current ETF price observations are available", call. = FALSE)
}


## Historically cheap


In [ ]:
df_etf_yahoo %>%
  group_by(series_id) %>%
  filter(date>=book_date()-years(5)) %>%
  dplyr::mutate(p_norm=pnorm(q = value,mean = mean(value),sd = sd(value))) %>%
  filter(date==max(date)) %>%
  ungroup() %>%
  arrange(p_norm) %>%
  dplyr::mutate(p_norm_label=scales::percent(p_norm,accuracy = 0.1)) %>%
  dplyr::select(series_id,p_norm_label) %>%
  inner_join(df_etfs,by=c("series_id"="Ticker")) %>%
  dplyr::select(series_id,p_norm_label,Name,Category,Subcategory,Yahoo.Finance.Link,Description) %>%
  datatable() # %>%
  # ggplot(.,aes(x=Name,y=p_norm)) +
  # geom_col() +
  # facet_wrap(~Category)


## Data sources and preparation {#data-1}


In [ ]:
# BYGV80_meta <- book_dst_metadata(table = "BYGV80", lang = "da")
BYGV80 <- book_dst(table = "BYGV80",
                       BYGFASE="*",
                       ANVENDELSE="*",
                       Tid="*",
                       lang = "da") %>%
  group_by(BYGFASE,ANVENDELSE) %>%
  arrange(BYGFASE,ANVENDELSE,TID) %>%
  filter(ANVENDELSE %in% c("Parcelhuse")) %>%
  ungroup() %>%
  dplyr::select(BYGFASE,
                TID,
                value) %>%
  dplyr::rename(series_id=BYGFASE,
                date=TID)


# BYG1_meta <- book_dst_metadata(table = "BYG1", lang = "da")
BYG1 <- book_dst(table = "BYG1",
                       BRANCHE07="*",
                       SÆSON="*",
                       ART="*",
                       Tid="*",
                       lang = "da") %>%
  filter(SÆSON=="Sæsonkorrigeret") %>%
  dplyr::mutate(filler="BYG1") %>%
  unite(series_id,filler,BRANCHE07,ART,sep = "_") %>%
  dplyr::select(series_id,
                TID,
                value) %>%
  dplyr::rename(date=TID)

# PRIS90_meta <- book_dst_metadata(table = "PRIS90", lang = "da")
PRIS90 <- book_dst(table = "PRIS90",
                       ENHED="*",
                       BOLTYP="*",
                       Tid="*",
                       lang = "da") %>%
  dplyr::rename(date=TID) %>%
  filter(ENHED=="Indeks")  %>%
  unite(BOLTYP,BOLTYP,ENHED) %>%
  dplyr::rename(series_id=BOLTYP)

# BYG42_meta <- book_dst_metadata(table = "BYG42", lang = "da")
BYG42 <- book_dst(table = "BYG42",
                       HINDEKS="*",
                       DINDEKS="*",
                       ART="*",
                       TAL="*",
                       Tid="*",
                       lang = "da") %>%
  dplyr::rename(date=TID) %>%
  filter(DINDEKS=="Byggeomkostningsindeks i alt") %>%
  filter(TAL=="Indeks") %>%
  filter(ART %in% c("Materialer","Arbejdsomkostninger")) %>%
  dplyr::mutate(filler="BYG42") %>%
  unite(series_id,filler,HINDEKS,DINDEKS,ART,sep="_")%>%
  dplyr::select(series_id,
                date,
                value)


## FinansDanmark


In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"
xlsx_url = finansdanmark_xlsx_url(source_url)

df_interest = book_workbook(url = xlsx_url,startRow = 1)
df_interest$År[1] = 1997
df_interest = df_interest %>% fill(År,.direction = "down")
df_interest$Date = book_iso_week_monday(df_interest$År, df_interest$Uge)
df_interest = df_interest %>%
  dplyr::select(Date,Kort.rente,Lang.rente) %>%
  dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente ) %>%
  gather(series_id,value,CurveLongShort,Lang.rente,Kort.rente) %>%
  dplyr::rename(date=Date) %>%
  dplyr::select(date,value,series_id)


## FRED


In [ ]:
series_ids = unique(c("CBBTCUSD",
               "NASDAQCOM",
               "T10Y2Y",
               "T5YIFR",
               "BAA10Y",
               "T10Y3M",
               "CPIAUCSL",
               "DCOILWTICO",
               "DFF",
               "ACTLISCOUUS",
               "MORTGAGE30US",
               "COMPUTSA",
               "RRVRUSQ156N",
               "UMCSENT",
               "BAMLH0A0HYM2",
               "RHORUSQ156N",
               "T10YIE",
               "HOUST1F",
               "BAMLH0A0HYM2",
               "COMPU1USA",
               "M2V",
               "BAMLC0A4CBBB",
               "PCEPI",
               "AUTHNOTT",
               "PSAVERT",
               "RRPONTSYD",
               "PERMIT",
               "M1SL",
               "PERMIT1",
               "T10Y3M",
               "INDPRO",
               "BAMLC0A4CBBB",
               "NFCI",
               "CORESTICKM159SFRBATL",
               "PAYEMS",
               "CIVPART",
               "MDSP",
               "M2SL",
               "UNRATE",
               "AUTHNOT1U",
               "HOUST",
               "ICSA",
               "DGS10",
               "DGS20",
               "DGS30",
               "DGS5",
               "DGS2",
               "DGS1",
               "CSUSHPINSA",
               "GDP",
               "GDPC1",
               "DFII10",
               "TDSP",
               "WALCL",
               "MSPUS"))

df_macro_fred = rbindlist(lapply(series_ids,function(series_id) {
  book_fred(series_id = series_id) %>%
  dplyr::select(date,value) %>%
    dplyr::mutate(series_id=series_id) #%>%
    # tidyr::complete(date=seq.Date(from=min(date,na.rm=T),to=book_date(),by="1 day")) %>%
    # tidyr::fill(value,.direction = "down")  %>%
    # tidyr::fill(series_id,.direction = "down")
}))


## Yahoo


In [ ]:
series_ids_yahoo = unique(c("AAPL",
                            "JNJ",
                            "MSFT",
                            "GC=F",
                            "SI=F",
                            "DX-Y.NYB",
                            # "BTC-USD",
                            # "ETH-USD",
                            "PA=F",
                            "^IXIC",
                            "^GSPC",
                            "^OMXC25",
                            "USDDKK=X"))

df_macro_yahoo = rbindlist(lapply(series_ids_yahoo, get_yahoo_value), fill = TRUE) %>%
  bind_rows( df_etf_yahoo )


## Assemble ETF series {#combine-2}

## Validate observations and remove identical repeats


In [ ]:
df_all = df_macro_yahoo %>%
  bind_rows( df_macro_fred ) %>%
  bind_rows(df_interest) %>%
  bind_rows(BYG42) %>%
  bind_rows(PRIS90) %>%
  bind_rows(BYGV80)
df_all = book_prepare_correlations(df_all, namespace = book_correlation_namespace)
all_ids = unique(df_all$series_id)

rm(df_macro_yahoo)
rm(df_macro_fred)
rm(df_etf_yahoo)
gc()


## Align every directed ETF comparison {#combine-3}


In [ ]:
# Alignment matches shared dates and omits pairs of missing values, without
# adding dates, filling observations, changing prices, or resampling series.


## Cross correlations

Cross correlation measures association, not causation. A leading lag does not establish a causal effect or a reliable forecast. The 30, 365 and 1,825 horizons count aligned observations, which are not necessarily calendar days for mixed-frequency sources. Comparisons with insufficient overlap or constant values remain in the results with undefined (`NA`) correlations.


In [ ]:
df_ccfs = book_correlations(df_all, namespace = "etf-correlations") %>%
  dplyr::mutate(best_corr_time = best_corr_time_365,
                best_corr_value = best_corr_value_365)
book_capture_correlation(book_correlation_namespace, df_all, df_ccfs, df_etfs$Ticker)


In [ ]:
df_ccfs %>%
  ggplot(.,aes(x=best_corr_time,y=best_corr_value)) +
  geom_point() +
  geom_vline(xintercept = 0,linetype=2) +
  geom_hline(yintercept = 0,linetype=2) +
  scale_y_continuous(limits = c(-1,1),breaks = seq(-1,1,by=0.1)) +
  scale_x_continuous(breaks = seq(-10,10,by=1)*365,labels=seq(-10,10,by=1))


## Descriptive

## Most dominating current indicators


In [ ]:
df_threshold_indicators = do.call("rbind",lapply(seq(0,1,by=0.01),function(threshold)  {
  df_ccfs %>%
    filter(abs(best_corr_value)>=threshold) %>%
    group_by(x1) %>%
    dplyr::summarize(count=n()) %>%
    ungroup() %>%
    dplyr::mutate(threshold=threshold)
}))

df_threshold_indicators %>%
  filter(threshold>0.95) %>%
  arrange(desc(count))

df_threshold_indicators %>%
  filter(threshold>0.95) %>%
  ggplot(.,aes(x=threshold,y=count,color=x1)) +
  geom_line() +
  theme(legend.position="null")


## Export


In [ ]:
df_etfs %>%
  inner_join(df_ccfs,by=c("Ticker"="x1")) %>%
  filter(abs(best_corr_value)>0.8) %>%
  filter(x2 %in% series_ids) %>%
  dplyr::select(Ticker,
                Name,
                Category,
                Subcategory,
                x2,
                best_corr_value) %>%
  dplyr::mutate(best_corr_value=scales::number(best_corr_value,accuracy=0.01)) %>%
  arrange(Ticker,desc(best_corr_value)) %>%
  datatable()

# openxlsx::write.xlsx(df_export,file = "etf_correlations.xlsx")


## Association graph


In [ ]:
# Load required libraries
library(igraph)

# Create a data frame representing strong associations
association_structure = df_ccfs %>%
  filter(abs(best_corr_value)>0.9) %>%
  dplyr::rename(from=x1,
                to=x2) %>%
  dplyr::mutate(sign=ifelse(best_corr_value>0,"positive","negative"),
                strength=scales::rescale(abs(best_corr_value),to = c(0,1),from = c(0,1))) %>%
  dplyr::select(from,to,sign,strength)

# Convert the data frame to edge_directions
edge_directions <- as.vector(t(association_structure[, c("from", "to")]))

# Create a graph object with edge_directions
g <- graph(edge_directions, directed = TRUE)

# Create a named vector of colors for positive and negative relationships
edge_colors <- c("positive" = "blue", "negative" = "red")

# Map the colors and thickness using the observed sign and strength
E(g)$color <- edge_colors[association_structure$sign]

# Normalize the strength values and map them to edge.width
normalize_strength <- function(strength, min_width, max_width) {
  normalized_strength <- (strength - min(strength)) / (max(strength) - min(strength))
  arrow_width <- normalized_strength * (max_width - min_width) + min_width
  return(arrow_width)
}

normalized_strength <- normalize_strength(association_structure$strength, min_width = 0.5, max_width = 2)
E(g)$width <- normalized_strength

# Plot the association graph with colored and thickness-scaled arrows
plot(g, vertex.label.family = "sans", vertex.label.font = 2, vertex.label.cex = 0.6, edge.arrow.size = 0.2)


In [ ]:
# Load required libraries
library(visNetwork)
library(dplyr)

corr_extra = data.frame(x1=all_ids,
                        x2=all_ids,
                        best_corr_value=1)

filtered_cor_df <- df_ccfs %>%
  ungroup() %>%
  filter(x1 %in% c("Lang.rente","Kort.rente")) %>%
  filter(abs(best_corr_value) > 0.7)

# Create a nodes data frame
nodes <- data.frame(id = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)), label = unique(c(filtered_cor_df$x1, filtered_cor_df$x2)))

# Create an edges data frame
edges <- filtered_cor_df %>%
  mutate(color = if_else(best_corr_value > 0, "green", "red"),
         width = scales::rescale(abs(best_corr_value),to=c(0,30))) %>%
  dplyr::select(from = x1, to = x2, color, width)

# Create a visNetwork graph
visNetwork(nodes, edges) %>%
  visIgraphLayout() %>%
  visOptions(highlightNearest = TRUE, nodesIdSelection = TRUE) %>%
  visInteraction(navigationButtons = TRUE)


## Bitcoin


In [ ]:
df_ccfs %>%
  filter(x1=="CBBTCUSD") %>%
  dplyr::mutate(best_corr_value=scales::number(best_corr_value,accuracy=0.01)) %>%
  datatable()


## Short housing


In [ ]:
df_ccfs %>%
  filter(x1=="CSUSHPINSA") %>%
  dplyr::mutate(best_corr_value=scales::number(best_corr_value,accuracy=0.01)) %>%
  datatable()


## Short DFF


In [ ]:
df_ccfs %>%
  filter(x1=="DFF") %>%
  dplyr::mutate(best_corr_value=scales::number(best_corr_value,accuracy=0.01)) %>%
  datatable()


## Short Treasury


In [ ]:
df_ccfs %>%
  filter(x1=="DGS30") %>%
  dplyr::mutate(best_corr_value=scales::number(best_corr_value,accuracy=0.01)) %>%
  datatable()
